# Day 29 — Correlation
### Statistics & Probability · Module 2 · Topic 2.9

**Prepared & presented by Srinivasa Sai Chava**  ·  Boston University

---

**Session length:** 2 hours · 90 min concepts + 30 min practice

| # | What we cover | Time |
|---|---|---|
| 1 | What r measures — and the two things it does not | 15 min |
| 2 | ⚠️ **What r cannot see** — Anscombe, curves, outliers | 25 min |
| 3 | How well do you actually know r? | 15 min |
| 4 | ⚠️ **The four explanations** — only one of them is cause | 25 min |
| 5 | Searching for correlations, and what to report | 10 min |
| 6 | **Practice notebook (separate file)** | 30 min |

---

## The step we take today

Yesterday you learned to ask *"is this difference real?"*. Today you ask a different
question: *"do these two things move together?"*

The answer is one number, **Pearson's r**, which runs from −1 to +1. The formula is small.
You could learn it in five minutes.

### So why does this need two hours?

Because everybody already knows the slogan *"correlation is not causation"* — and almost
nobody can name the four reasons two things can correlate. The slogan tells you what the
answer is **not**. It does not tell you what else could be going on, so people repeat it
and then quietly believe the causal story anyway.

### Four claims to hold me to

1. Four completely different pictures will produce the identical **r = 0.816**.
2. One extra data point will turn **r = −0.14** into **r = +0.64**, with p = 0.00000000000047.
3. Two variables with **zero** relationship will correlate at **−0.73**.
4. Women will be admitted at a higher rate in **every** department, and a lower rate overall.

All four are computed in this notebook. Nothing below is a stylised cartoon.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.simplefilter("ignore")

plt.rcParams["figure.figsize"] = (8, 3.2)
print("numpy", np.__version__, "| pandas", pd.__version__, "| scipy", __import__("scipy").__version__)

---
# 1 · What r measures

## 1.1 · The idea in one sentence

**Pearson's r measures how tightly a cloud of points hugs a straight line.**

That is the whole job. Not whether there is a relationship — whether there is a *straight*
one. Not how big the effect is — how *predictable* it is.

The formula standardises both variables and then averages their product:

$$r \;=\; \frac{1}{n-1}\sum_{i=1}^{n} \left(\frac{x_i - \bar{x}}{s_x}\right)\left(\frac{y_i - \bar{y}}{s_y}\right)$$

Read it as: *when x is above its average, is y usually above its average too?* If yes, every
product is positive and r is positive. If x being high goes with y being low, the products
are negative. If there is no pattern, the positives and negatives cancel and r lands near 0.

Because both variables are divided by their own standard deviation, **the units cancel out**.
That is exactly why r cannot tell you how much y changes — we come back to this in 1.3.

In [ ]:
# The formula, written out, next to scipy - they agree exactly
def r_by_hand(x, y):
    x, y = np.asarray(x, float), np.asarray(y, float)
    zx = (x - x.mean()) / x.std(ddof=1)
    zy = (y - y.mean()) / y.std(ddof=1)
    return (zx * zy).sum() / (len(x) - 1)

rng = np.random.default_rng(7)
x = rng.normal(0, 1, 50)
y = 0.8*x + rng.normal(0, 0.6, 50)

print("by hand :", round(r_by_hand(x, y), 10))
print("scipy   :", round(stats.pearsonr(x, y).statistic, 10))
print("numpy   :", round(np.corrcoef(x, y)[0, 1], 10))

## 1.2 · What the values look like

Before you trust any rule of thumb, you should know what these numbers look like on a
scatter plot. Most people badly **overestimate** weak correlations and **underestimate**
strong ones.

Guess each panel before you read the title.

In [ ]:
def corr_data(r, n=200, seed=0):
    """Draw n points with (population) correlation exactly r."""
    rng = np.random.default_rng(seed)
    x = rng.normal(0, 1, n)
    y = r*x + np.sqrt(max(0, 1 - r**2))*rng.normal(0, 1, n)
    return x, y

fig, axes = plt.subplots(2, 4, figsize=(13, 5.2))
for ax, r in zip(axes.ravel(), [0.0, 0.3, 0.6, 0.85, -0.3, -0.6, -0.85, 0.99]):
    xx, yy = corr_data(r, seed=int(abs(r)*100) + 1)
    ax.scatter(xx, yy, s=10, alpha=.75)
    ax.set_title(f"r = {stats.pearsonr(xx, yy).statistic:+.2f}", fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("What r actually looks like", fontsize=13)
plt.tight_layout(); plt.show()

Notice the top-left pair. **r = 0.3** sounds like "some relationship", and it looks like
almost nothing. Hold that thought until section 1.4.

## 1.3 · ⚠️ r is not the slope

This is the single most common misuse of r in practice: someone reports r when the question
needed the slope.

Below are four datasets. Their fitted slopes run from about **0.09** to about **90** — a
thousandfold difference in how much y changes per unit of x. Their r is **identical**.

In [ ]:
rng = np.random.default_rng(291)
base_x = rng.normal(0, 1, 300)
noise  = rng.normal(0, 1, 300)

rows = []
for s in [0.1, 1, 10, 100]:
    y = s*base_x + noise*s          # signal AND noise both scaled - shape unchanged
    slope = np.polyfit(base_x, y, 1)[0]
    rows.append({"nominal slope": s,
                 "fitted slope": round(slope, 4),
                 "r": round(stats.pearsonr(base_x, y).statistic, 4)})
print(pd.DataFrame(rows).to_string(index=False))

Same r, every time. Scaling y by any positive number leaves r untouched, because r is
computed **after** both variables are divided by their own standard deviation.

| Question | The right number |
|---|---|
| "How predictable is y from x?" | **r** (or r²) |
| "How much does y change per unit of x?" | **the slope** |
| "Is the relationship real, or noise?" | **the p-value and the interval** |

If someone asks *"how much does revenue rise per extra salesperson?"*, r cannot answer it.
That is the regression slope, and it is Module 3's main business.

## 1.4 · r² — the number that keeps you honest

Square r and it becomes interpretable: **r² is the share of the variation in y that x
accounts for.**

This is the number to quote when somebody sounds impressed by a correlation.

In [ ]:
tbl = pd.DataFrame({"r": [0.1, 0.3, 0.5, 0.7, 0.9]})
tbl["r2"] = tbl["r"]**2
tbl["share of variation in y"] = (tbl["r2"]*100).round(0).astype(int).astype(str) + "%"
tbl["often called"] = ["essentially nothing", "weak", "moderate", "strong", "very strong"]
print(tbl[["r", "r2", "share of variation in y", "often called"]].to_string(index=False))

**The honest habit: whenever you report an r, report r² beside it.**

> *"Hours studied correlates with marks at r = 0.3"*
> sounds like a finding worth acting on.
>
> *"Hours studied accounts for 9% of the variation in marks"*
> is the same fact, and it is the version that stops anyone redesigning a syllabus around it.

⚠️ Those adjectives are **conventions, not laws**. In physics r = 0.9 is often disappointing;
in psychology r = 0.3 can be a strong, replicable result. What counts as "strong" depends
entirely on the field and on how noisy the measurement is.

---
# 2 · ⚠️ What r cannot see

Section 1 was the easy half. This is where r starts lying to you.

## 2.1 · ⭐ Anscombe's quartet

Four datasets. Identical mean of x, identical mean of y, identical standard deviations,
identical regression line, identical r.

Francis Anscombe built them in 1973 to make a single point. Look at the numbers first.

In [ ]:
x1 = np.array([10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5], float)
y1 = np.array([8.04, 6.95, 7.58, 8.81, 8.33, 9.96, 7.24, 4.26, 10.84, 4.82, 5.68])
y2 = np.array([9.14, 8.14, 8.74, 8.77, 9.26, 8.10, 6.13, 3.10, 9.13, 7.26, 4.74])
y3 = np.array([7.46, 6.77, 12.74, 7.11, 7.81, 8.84, 6.08, 5.39, 8.15, 6.42, 5.73])
x4 = np.array([8, 8, 8, 8, 8, 8, 8, 19, 8, 8, 8], float)
y4 = np.array([6.58, 5.76, 7.71, 8.84, 8.47, 7.04, 5.25, 12.50, 5.56, 7.91, 6.89])

sets = [("I", x1, y1), ("II", x1, y2), ("III", x1, y3), ("IV", x4, y4)]
rows = []
for name, xx, yy in sets:
    slope, icept = np.polyfit(xx, yy, 1)
    rows.append({"set": name,
                 "mean x": round(xx.mean(), 2), "mean y": round(yy.mean(), 2),
                 "sd y": round(yy.std(ddof=1), 2),
                 "slope": round(slope, 3), "intercept": round(icept, 3),
                 "r": round(stats.pearsonr(xx, yy).statistic, 3)})
print(pd.DataFrame(rows).to_string(index=False))

Every column agrees to the precision anybody would ever report. (The four r values are
0.8164, 0.8162, 0.8163 and 0.8165 — they differ in the **fourth** decimal place, which is
why set IV shows as 0.817 above.) Any summary table you could put in a report would say
these four datasets are the same.

Now plot them.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(13, 3.1))
for ax, (name, xx, yy) in zip(axes, sets):
    ax.scatter(xx, yy, s=38, zorder=3, edgecolor="white", lw=1)
    g = np.linspace(2, 20, 10)
    ax.plot(g, np.polyval(np.polyfit(xx, yy, 1), g), lw=2)
    ax.set_xlim(2, 20); ax.set_ylim(2, 14)
    ax.set_title(f"set {name}   r = 0.816", fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("Same numbers. Not the same data.", fontsize=13)
plt.tight_layout(); plt.show()

- **Set I** is an honest linear relationship — the only one you would want to model this way.
- **Set II** is a clean curve. Fitting a line to it is simply wrong.
- **Set III** is a perfect line plus **one** outlier, which drags the fit off course.
- **Set IV** is a vertical stack plus **one** point. That single point creates the entire
  correlation; delete it and r is undefined.

> **The rule that follows: plot the data. Every time. It takes four seconds.**

## 2.2 · r only ever asks about straight lines

Anscombe set II hinted at it; here it is in full. In each panel below, x pins y down to one
or two values — these are near-deterministic relationships. Pearson's r reports nothing.

In [ ]:
rng = np.random.default_rng(298)
cases = []
xa = np.linspace(-3, 3, 200)
cases.append((xa, xa**2 + rng.normal(0, 0.6, 200), "a perfect U"))
xb = np.linspace(0, 2*np.pi, 200)
cases.append((xb, np.cos(xb) + rng.normal(0, 0.12, 200), "a wave"))
xc = rng.uniform(-3, 3, 220)
cases.append((xc, xc*rng.choice([-1, 1], 220) + rng.normal(0, 0.12, 220), "a perfect X"))
ang = rng.uniform(0, 2*np.pi, 250)
cases.append((np.cos(ang) + rng.normal(0, .06, 250),
              np.sin(ang) + rng.normal(0, .06, 250), "a ring"))

fig, axes = plt.subplots(1, 4, figsize=(13, 3.0))
for ax, (xx, yy, t) in zip(axes, cases):
    ax.scatter(xx, yy, s=10, alpha=.85)
    ax.set_title(f"{t}\nr = {stats.pearsonr(xx, yy).statistic:+.3f}", fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("x tells you y. r reports nothing.", fontsize=13)
plt.tight_layout(); plt.show()

Every r comes back under 0.07.

**This is not a toy problem.** U-shaped relationships are the normal case in large parts of
the world:

- **dose and effect** — too little does nothing, too much harms
- **price and demand** for a luxury good — cheap reads as low quality
- **practice and performance** — gains flatten, then over-training hurts
- **age and almost any outcome** — income, risk tolerance, reaction time

Run `df.corr()` across a table full of those and you will read "no relationship" off every
one of them.

## 2.3 · Spearman — the fix for curves that only go one way

**Spearman's ρ is just Pearson's r computed on the *ranks* instead of the values.**

Replace every value by its position in the sorted order (smallest = 1, next = 2, …) and
compute r on those. Any relationship that only ever goes **one direction** — monotonic —
straightens out completely.

It asks: *when x goes up, does y go up?*
Not: *does y go up in a straight line?*

In [ ]:
rng = np.random.default_rng(299)
x = np.linspace(1, 10, 120)
y = np.exp(x/2) * np.exp(rng.normal(0, 0.09, 120))     # a steep but monotonic curve

rx, ry = stats.rankdata(x), stats.rankdata(y)

fig, axes = plt.subplots(1, 2, figsize=(9.2, 3.2))
axes[0].scatter(x, y, s=16)
axes[0].set_title(f"raw values\nPearson r = {stats.pearsonr(x, y).statistic:.3f}", fontsize=11)
axes[0].set_xlabel("x"); axes[0].set_ylabel("y")
axes[1].scatter(rx, ry, s=16, color="seagreen")
axes[1].set_title(f"the same data, ranked\nr of the ranks = {stats.pearsonr(rx, ry).statistic:.3f}",
                  fontsize=11)
axes[1].set_xlabel("rank of x"); axes[1].set_ylabel("rank of y")
plt.tight_layout(); plt.show()

print("stats.spearmanr  :", round(stats.spearmanr(x, y).statistic, 4))
print("pearson on ranks :", round(stats.pearsonr(rx, ry).statistic, 4))

Pearson saw 0.877 on a relationship that is essentially perfectly monotonic. Spearman sees
0.997 and gets it right. (They are the same calculation — `spearmanr` just ranks first.)

**Reach for Spearman when:**

- the relationship is curved but always goes one way
- there are heavy outliers you cannot justify removing
- the data is **ordinal** — star ratings, grades, Likert scales, finishing positions

⚠️ **Spearman is not a general fix.** It cannot see a U-shape either: a U goes up and then
down, so the ranks do not line up any better than the values did. Check for yourself:

In [ ]:
u_x = np.linspace(-3, 3, 200)
u_y = u_x**2
print("perfect U -> Pearson :", round(stats.pearsonr(u_x, u_y).statistic, 4))
print("perfect U -> Spearman:", round(stats.spearmanr(u_x, u_y).statistic, 4))
print("\nBoth near zero on a relationship with NO noise in it at all.")

## 2.4 · ⚠️ One point can manufacture a correlation, or erase one

Pearson's r is built from squared distances, so a single far-away point carries enormous
weight. This is not a rare pathology — it is a stray test record, a value in the wrong
units, a missing code stored as 999, a decimal point in the wrong place.

In [ ]:
# --- one point INVENTS a relationship -------------------------------
rng = np.random.default_rng(292)
x = rng.normal(50, 10, 100)
y = rng.normal(50, 10, 100)                 # genuinely unrelated
xo, yo = np.append(x, 200), np.append(y, 200)   # one typo at (200, 200)

before = stats.pearsonr(x, y)
after  = stats.pearsonr(xo, yo)
print("100 unrelated points      : r = %+.4f  p = %.3f" % (before.statistic, before.pvalue))
print("+ ONE point at (200, 200) : r = %+.4f  p = %.2e" % (after.statistic, after.pvalue))
print("Spearman, before -> after : %+.4f -> %+.4f"
      % (stats.spearmanr(x, y).statistic, stats.spearmanr(xo, yo).statistic))

In [ ]:
# --- one point DESTROYS a relationship ------------------------------
rng = np.random.default_rng(293)
x2 = rng.normal(0, 1, 100)
y2 = x2 + rng.normal(0, 0.3, 100)           # a genuinely strong relationship
x3, y3 = np.append(x2, 12), np.append(y2, -12)

print("a strong real relationship : r = %+.4f" % stats.pearsonr(x2, y2).statistic)
print("+ ONE bad point            : r = %+.4f" % stats.pearsonr(x3, y3).statistic)
print("Spearman, before -> after  : %+.4f -> %+.4f"
      % (stats.spearmanr(x2, y2).statistic, stats.spearmanr(x3, y3).statistic))

Read the first block again: **p = 4.7e-13**. A tiny p-value is no protection against a
data-quality problem. The test is perfectly correct — it is answering a question about data
that contains a typo.

Notice what Spearman does in both cases: it **barely moves**. A rank cannot run away. One
value being 200 instead of 60 only makes it the largest rank, which it already was.

> **The cheapest outlier detector you will ever write: compute both, and look at the gap.**
> If Pearson and Spearman disagree by more than about 0.15, something in the data deserves
> a look.

---
# 3 · How well do you actually know r?

## 3.1 · r is an estimate, and estimates have intervals

`stats.pearsonr` returns a single number to 15 decimal places, which makes it very easy to
forget that it is an **estimate from a sample**. Draw a different sample and you get a
different r.

Suppose you observe **r = 0.50**. What is the true correlation? It depends enormously on
how many points you had.

In [ ]:
def r_interval(r, n, conf=0.95):
    """Fisher z interval for a correlation - what scipy's confidence_interval() does."""
    z  = np.arctanh(r)
    se = 1/np.sqrt(n - 3)
    crit = stats.norm.ppf(1 - (1 - conf)/2)
    return np.tanh(z - crit*se), np.tanh(z + crit*se)

print("observed r = 0.50\n")
print(f"{'n':>7} | {'95% interval for the TRUE correlation':^38} | width")
print("-"*58)
for n in [10, 20, 30, 50, 100, 200, 500]:
    lo, hi = r_interval(0.50, n)
    print(f"{n:>7} | {f'[{lo:+.3f}, {hi:+.3f}]':^38} | {hi-lo:.3f}")

Read the first row. **r = 0.50 from 10 points is compatible with the true correlation being
anywhere from −0.19 to +0.86.** It includes zero *and* it includes almost perfect. You have
learned essentially nothing, and the number "0.50" hides that completely.

scipy gives you the interval directly — one extra line:

In [ ]:
rng = np.random.default_rng(55)
xs, ys = corr_data(0.5, n=10, seed=55)
res = stats.pearsonr(xs, ys)
lo, hi = res.confidence_interval()
print("n = 10   r = %+.3f   95%% CI [%+.3f, %+.3f]   p = %.3f"
      % (res.statistic, lo, hi, res.pvalue))

xs, ys = corr_data(0.5, n=500, seed=55)
res = stats.pearsonr(xs, ys)
lo, hi = res.confidence_interval()
print("n = 500  r = %+.3f   95%% CI [%+.3f, %+.3f]   p = %.2e"
      % (res.statistic, lo, hi, res.pvalue))

⚠️ **Below about n = 30, treat any correlation as a hint rather than a measurement** — and
always print the interval beside it.

## 3.2 · ⚠️ The other end: a meaningless r is "significant" at large n

Yesterday's lesson in a new costume. The p-value for a correlation depends on **n** as much
as on **r**, so in a large dataset almost anything clears p < 0.05.

In [ ]:
def smallest_significant_r(n, alpha=0.05):
    """The smallest |r| that reaches p < alpha at this sample size."""
    t = stats.t.ppf(1 - alpha/2, n - 2)
    return np.sqrt(t**2 / (t**2 + n - 2))

print(f"{'n':>8} | {'smallest significant r':>22} | {'r^2':>10}")
print("-"*48)
for n in [10, 20, 50, 100, 500, 1_000, 10_000, 100_000]:
    c = smallest_significant_r(n)
    print(f"{n:>8} | {c:>22.4f} | {c**2*100:>9.3g}%")

With 100,000 rows, any correlation above **0.006** is "statistically significant". That is
an r² of **0.004%** — a relationship so faint it is indistinguishable from nothing.

The p-value says *real*. The r² says *irrelevant*. **Both are right**, and only one of them
answers the question anybody actually cares about.

Put the two ends of this section together:

| | the problem | what fixes it |
|---|---|---|
| **small n** | r is wildly unstable | report the confidence interval |
| **large n** | the p-value stops carrying information | report r² |

Neither problem is solved by staring harder at r itself.

---
# 4 · ⚠️ The four explanations

Here is the section that is worth the whole session.

X and Y are correlated. There are **five** possible explanations, and only one of them is
"X causes Y":

| | explanation | example |
|---|---|---|
| 1 | **Chance** | you looked at 190 pairs and one came up |
| 2 | **Confounding** | something else causes both |
| 3 | **Reverse causation** | Y causes X, not the other way round |
| 4 | **Selection** | the way the data was collected created the link |
| 5 | **X really does cause Y** | the one everybody assumes |

The slide deck calls this "the four explanations" on purpose: the fifth is the one you walked
in already believing. Your job when you meet a correlation is to **argue about which of these
it is** — and that argument is not a calculation.

## 4.1 · Confounding — something else causes both

The classic: ice cream sales and drownings. Nobody has ever drowned from ice cream. Hot days
sell ice cream, and hot days put people in the water.

In [ ]:
rng = np.random.default_rng(294)
temp      = rng.uniform(5, 35, 365)                      # a year of daily temperatures
ice_cream = 20*temp + rng.normal(0, 60, 365)             # caused ONLY by temperature
drownings = 0.25*temp + rng.normal(0, 1.2, 365)          # also caused ONLY by temperature

r_naive = stats.pearsonr(ice_cream, drownings)
print("ice cream vs drownings : r = %+.3f   p = %.2e" % (r_naive.statistic, r_naive.pvalue))
print("temperature -> ice cream: r = %+.3f" % stats.pearsonr(temp, ice_cream).statistic)
print("temperature -> drownings: r = %+.3f" % stats.pearsonr(temp, drownings).statistic)

r = 0.82 with p = 2.7e-91. That is about as strong and as significant as correlations get
in real data, and the causal story is nonsense.

**The fix: hold the confounder still.** Compare ice cream and drownings *within* days of the
same temperature. That is a **partial correlation**: regress each variable on the confounder,
then correlate what is left over.

In [ ]:
def partial_corr(a, b, control):
    """Correlation between a and b with the linear effect of `control` removed."""
    resid_a = a - np.polyval(np.polyfit(control, a, 1), control)
    resid_b = b - np.polyval(np.polyfit(control, b, 1), control)
    return stats.pearsonr(resid_a, resid_b)

pc = partial_corr(ice_cream, drownings, temp)
print("raw correlation                      : %+.3f" % r_naive.statistic)
print("partial correlation, controlling temp: %+.3f   (p = %.3f)" % (pc.statistic, pc.pvalue))

From 0.823 to 0.048. Once you know the temperature, ice cream tells you nothing more about
drownings.

⚠️ This only works for confounders **you thought of and measured**. There is no procedure
that controls for a variable you did not collect, and no amount of data fixes that. This is
the whole reason randomised experiments exist: randomisation balances the confounders you
never even imagined.

## 4.2 · Reverse causation — the arrow points the other way

A correlation is **symmetric**: `r(x, y)` and `r(y, x)` are the same number, exactly. The
direction of causation is simply not in the data.

In [ ]:
rng = np.random.default_rng(11)
a = rng.normal(0, 1, 200)
b = 0.7*a + rng.normal(0, 0.7, 200)
print("r(a, b) =", round(stats.pearsonr(a, b).statistic, 12))
print("r(b, a) =", round(stats.pearsonr(b, a).statistic, 12))
print("\nIdentical. The data has no opinion about which causes which.")

Four real claims where the arrow is probably backwards:

| the correlation | the likely direction |
|---|---|
| Hospitals with more staff have more deaths | bigger hospitals take sicker patients |
| People who diet weigh more | weighing more is *why* they started dieting |
| Firms using more consultants perform worse | struggling firms hire consultants |
| Heavier police presence, more crime | crime draws police to the area |

**The test:** ask whether the story works just as well backwards. If it does, your data
cannot distinguish them, and you need either a clear time ordering or an experiment.

## 4.3 · ⭐ Selection — when collecting the data creates the link

This is the one almost no introductory course teaches, and it is the one that quietly ruins
analyses of customer, patient, admissions and hiring data.

Suppose talent and good looks are **completely independent** in the population — which they
are, in this simulation, by construction. But you become famous if you have enough of
*either*. Now look only at famous people.

In [ ]:
rng = np.random.default_rng(295)
N = 3000
talent = rng.normal(0, 1, N)
looks  = rng.normal(0, 1, N)                 # independent BY CONSTRUCTION

famous = (talent + looks) > 1.8              # you need enough of one or the other

r_all = stats.pearsonr(talent, looks)
r_fam = stats.pearsonr(talent[famous], looks[famous])
print("everybody        : n = %4d   r = %+.3f   (p = %.2f)" % (N, r_all.statistic, r_all.pvalue))
print("only the famous  : n = %4d   r = %+.3f   (p = %.2e)"
      % (famous.sum(), r_fam.statistic, r_fam.pvalue))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].scatter(talent[~famous], looks[~famous], s=7, color="lightsteelblue")
axes[0].scatter(talent[famous], looks[famous], s=9, color="orangered")
axes[0].set_title(f"everybody\nr = {r_all.statistic:+.3f}  (no relationship)", fontsize=11)
axes[1].scatter(talent[famous], looks[famous], s=11, color="orangered")
g = np.linspace(talent[famous].min(), talent[famous].max(), 10)
axes[1].plot(g, np.polyval(np.polyfit(talent[famous], looks[famous], 1), g), color="black", lw=2)
axes[1].set_title(f"only the famous ones\nr = {r_fam.statistic:+.3f}", fontsize=11)
for ax in axes:
    ax.set_xlabel("talent"); ax.set_ylabel("looks")
plt.tight_layout(); plt.show()

**r = −0.726 between two variables that have no relationship whatsoever.** Nobody caused
anything. The *selection rule itself* wrote the correlation into the data.

The mechanism, in one sentence: fame is a **common effect** of talent and looks, and
conditioning on a common effect induces a correlation between its causes. If someone is
famous and not especially talented, they must be good-looking — otherwise they would not
be in the sample.

This is called **collider bias**, and it is lurking in any analysis restricted to people who
were **admitted, hired, diagnosed, subscribed, or survived**. Always ask: *who is in this
dataset, and what rule put them here?*

## 4.4 · ⚠️ Simpson's paradox — the reversal

The most unsettling one. A pattern that holds in **every** subgroup can **reverse** when you
pool the data. Based on the 1973 Berkeley graduate admissions case.

In [ ]:
# Four departments. Women are admitted at a HIGHER rate in every one of them.
data = pd.DataFrame({
    "dept":        ["A", "B", "C", "D"],
    "difficulty":  ["easy", "easy", "hard", "hard"],
    "men_applied": [800, 600, 150, 100],
    "men_rate":    [0.62, 0.58, 0.28, 0.22],
    "women_applied": [100, 120, 700, 650],
    "women_rate":    [0.68, 0.64, 0.32, 0.26],
})
data["men_admitted"]   = (data.men_applied   * data.men_rate).round().astype(int)
data["women_admitted"] = (data.women_applied * data.women_rate).round().astype(int)

view = data[["dept", "difficulty", "men_applied", "men_rate", "women_applied", "women_rate"]].copy()
view["women higher?"] = np.where(data.women_rate > data.men_rate, "YES", "no")
print(view.to_string(index=False))

In [ ]:
m_rate = data.men_admitted.sum()   / data.men_applied.sum()
w_rate = data.women_admitted.sum() / data.women_applied.sum()
print("\nOVERALL")
print("  men   : %d / %d = %.1f%%" % (data.men_admitted.sum(), data.men_applied.sum(), m_rate*100))
print("  women : %d / %d = %.1f%%" % (data.women_admitted.sum(), data.women_applied.sum(), w_rate*100))

easy = data.difficulty == "easy"
print("\nWHERE THEY APPLIED")
print("  %.0f%% of men applied to the easy departments"   % (data.men_applied[easy].sum()   / data.men_applied.sum()   * 100))
print("  %.0f%% of women applied to the easy departments" % (data.women_applied[easy].sum() / data.women_applied.sum() * 100))

Both statements are true, from the same table, with no arithmetic error anywhere:

- Women are admitted at a **higher** rate in **every single department**.
- Women are admitted at a **lower** rate **overall** (34.3% vs 55.0%).

The overall figure mostly measures **which doors people knocked on**, not what happened at
the door. 85% of the men applied to the easy departments; 14% of the women did.

⚠️ The lesson is **not** "always split by subgroup". Splitting by something else — surname
length, say — would be nonsense. The lesson is that **you must decide which grouping answers
your actual question**, and that decision is about the world, not about statistics. If the
question is "does this department discriminate?", look within departments. If the question is
"why do fewer women get in?", the answer is partly about which departments they applied to —
and then you have to ask why *that* is.

## 4.5 · The checklist

Six questions before you believe any correlation:

| # | ask | why |
|---|---|---|
| 1 | **Did I plot it?** | Anscombe. Non-negotiable, four seconds. |
| 2 | **How many pairs did I look at?** | if it was 190, this one means nothing alone |
| 3 | **What is r², and the interval?** | is it big, and do I actually know it? |
| 4 | **What else could cause both?** | name a specific confounder, then control for it |
| 5 | **Does the story work backwards?** | if yes, the data cannot tell you the direction |
| 6 | **Who is in this dataset, and why?** | selection: admitted, hired, survived, subscribed |

Questions 4, 5 and 6 **cannot be answered from the data**. They need you to know where the
numbers came from — which is why a domain expert beats a bigger dataset.

---
# 5 · Searching for correlations

## 5.1 · ⚠️ `df.corr()` on 20 columns runs 190 hypothesis tests

Yesterday's multiplicity lesson in its most common real-world form. A correlation heatmap
looks like one picture. It is $\binom{20}{2} = 190$ silent tests.

In [ ]:
rng = np.random.default_rng(296)
n_rows, n_cols = 40, 20
df = pd.DataFrame(rng.normal(0, 1, (n_rows, n_cols)),
                  columns=[f"var_{i:02d}" for i in range(n_cols)])
# Every column is independent noise. There is NOTHING here.

found = []
for i in range(n_cols):
    for j in range(i + 1, n_cols):
        res = stats.pearsonr(df.iloc[:, i], df.iloc[:, j])
        found.append((df.columns[i], df.columns[j], res.statistic, res.pvalue))

res_df = pd.DataFrame(found, columns=["a", "b", "r", "p"])
sig = res_df[res_df.p < 0.05].sort_values("p")

print("pairs tested        :", len(res_df))
print("p < 0.05            :", len(sig))
print("expected by chance  : %.1f" % (len(res_df) * 0.05))
print("\nThe 'best' finding:")
print(sig.head(3).to_string(index=False))

In [ ]:
bonf = 0.05 / len(res_df)
print("Bonferroni threshold for %d pairs : %.6f" % (len(res_df), bonf))
print("survivors                          :", (res_df.p < bonf).sum())
print("\nCorrect answer: there is nothing in this data, and nothing survived.")

Seven "significant" correlations out of pure noise, the strongest at r = −0.45 with
p = 0.003. In a report that would look like a finding.

> **A correlation heatmap is a tool for *generating* hypotheses. It can never *confirm* one.**
> If you found the pair by looking, you need either a correction or a fresh dataset.

## 5.2 · A full worked example

Study hours and exam scores, reported properly from start to finish.

In [ ]:
rng = np.random.default_rng(2901)
n = 80
hours = rng.gamma(4, 2.2, n)
score = np.clip(42 + 3.1*hours + rng.normal(0, 8, n), 0, 100)

# 1. PLOT IT FIRST
plt.figure(figsize=(7.5, 3.2))
plt.scatter(hours, score, s=28, alpha=.8)
slope, icept = np.polyfit(hours, score, 1)
g = np.linspace(hours.min(), hours.max(), 10)
plt.plot(g, slope*g + icept, lw=2, color="black")
plt.xlabel("hours studied"); plt.ylabel("exam score")
plt.tight_layout(); plt.show()

In [ ]:
# 2. Then the numbers - all five of them
pe = stats.pearsonr(hours, score)
sp = stats.spearmanr(hours, score)
lo, hi = pe.confidence_interval()

print("n        = %d" % n)
print("Pearson  = %+.3f   95%% CI [%+.3f, %+.3f]   p = %.2e" % (pe.statistic, lo, hi, pe.pvalue))
print("Spearman = %+.3f   (agrees -> not an outlier artefact)" % sp.statistic)
print("r^2      = %.1f%% of the variation in scores" % (pe.statistic**2 * 100))
print("slope    = %.2f marks per extra hour" % slope)

**What we can say:** hours studied and exam score are strongly related, r = 0.873, 95% CI
0.81 to 0.92, n = 80. Spearman agrees at 0.883, so it is not an outlier artefact. Hours
accounts for 76% of the variation in scores, and the slope is about 2.9 marks per extra hour.

⚠️ **What we cannot say:** that studying *causes* the marks. Work the checklist:

- **Confounding** — motivation plausibly drives both hours and marks.
- **Reverse** — students who find the material easy may study more because it is rewarding.
- **Selection** — this is one class. Who dropped out before the exam?

The sentence to actually write:

> *"Hours studied and exam score are strongly related (r = 0.87, 95% CI 0.81–0.92, n = 80;
> r² = 0.76). This is observational, so we cannot separate studying from the motivation that
> drives it."*

Notice how short the caveat is. Honesty about causation costs **one clause**, not a paragraph.

## 5.3 · Which coefficient, and what to report

| Situation | Use | scipy |
|---|---|---|
| Straight-line relationship, no wild outliers | Pearson | `stats.pearsonr(x, y)` |
| Curved but always one direction | Spearman | `stats.spearmanr(x, y)` |
| Heavy outliers you cannot remove | Spearman | `stats.spearmanr(x, y)` |
| Ordinal data (star ratings, grades) | Spearman or Kendall | `stats.kendalltau(x, y)` |
| Small sample with many ties | Kendall's tau | `stats.kendalltau(x, y)` |
| U-shaped or non-monotonic | **none of them — plot it** | `plt.scatter(x, y)` |
| Need the *size* of the effect | regression slope | `np.polyfit(x, y, 1)` |

**Always report together:** r, its 95% confidence interval, r², n, and whether you checked
Spearman. Five numbers, one line of output, and a reader can judge everything that matters.

---
# 6 · Putting it together

One function that does the whole checklist's computable half.

In [ ]:
def correlate(x, y, n_pairs=1, labels=("x", "y")):
    """Report a correlation with everything needed to judge it."""
    x, y = np.asarray(x, float), np.asarray(y, float)

    #  1. always plot. This single line prevents most mistakes.
    plt.figure(figsize=(5.5, 2.8))
    plt.scatter(x, y, s=18, alpha=.7)
    plt.xlabel(labels[0]); plt.ylabel(labels[1])
    plt.tight_layout(); plt.show()

    #  2. both coefficients - a gap between them means outliers
    pe = stats.pearsonr(x, y)
    sp = stats.spearmanr(x, y)
    lo, hi = pe.confidence_interval()

    print(f"n = {len(x)}")
    print(f"Pearson  r = {pe.statistic:+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]")
    print(f"Spearman r = {sp.statistic:+.3f}")
    print(f"r^2 = {pe.statistic**2:.1%} of the variance")
    print(f"slope = {np.polyfit(x, y, 1)[0]:+.3f} {labels[1]} per {labels[0]}")
    print(f"p = {pe.pvalue:.4f}   threshold {0.05/n_pairs:.5f} "
          f"({n_pairs} pair(s) examined)")
    if abs(pe.statistic - sp.statistic) > 0.15:
        print("  !! Pearson and Spearman disagree - check for outliers")

In [ ]:
# on the honest dataset
correlate(hours, score, n_pairs=1, labels=("hour", "marks"))

In [ ]:
# on the dataset with one typo in it - watch the warning fire
correlate(xo, yo, n_pairs=1, labels=("x", "y"))

The warning fires on the second one. What `correlate()` **cannot** do is tell you whether X
causes Y. No function can. That is questions 4, 5 and 6 of the checklist, and they need you.

---
# Recap — today in twelve lines

1. **r** runs from −1 to +1 and measures one thing: tightness around a **straight** line.
2. ⚠️ **r is not a slope.** Slopes of 0.09 and 90 both gave r = 0.697.
3. **r²** is the share of variance. r = 0.3 is 9%, which is almost nothing.
4. ⭐ **Anscombe:** four different pictures, all with r = 0.816. **Plot the data.**
5. ⚠️ r sees only straight lines. A perfect U-shape gives r ≈ 0.
6. **Spearman** = r on the ranks. Fixes monotonic curves and outliers, not U-shapes.
7. ⚠️ One point moved r from −0.14 to +0.64, at p = 4.7e-13.
8. ⚠️ r = 0.5 from n = 10 has a 95% CI of −0.19 to +0.86. You learned nothing.
9. ⚠️ At n = 100,000, r = 0.006 is "significant" and explains 0.004%.
10. **Five explanations:** chance, confounding, reverse, selection — and cause.
11. ⚠️ Selection manufactured r = −0.73 between two independent variables.
12. ⚠️ `df.corr()` on 20 columns is **190 hypothesis tests**, not one picture.

## Homework

- Run `correlate()` on three pairs of columns from your Day 21 project and report all five
  numbers for each.
- Find a news headline built on a correlation and work the six-question checklist on it.
- Take one correlation you believe is causal and name the specific confounder you would need
  to rule out.

### Next class — Topic 2.10: the statistics mini project
Everything from Day 23 onwards, applied end to end on a real dataset — describe it, test
something, and report it honestly enough that someone could check you.

---
*Slides & notebooks by Srinivasa Sai Chava · Boston University*